# Heart Disease Prediction with Tree Ensembles

## Goals
Train a Decision Tree, a Random Forest and an XGBoost model on real patient data to predict heart disease, compare how each one overfits, and save the best model for predictions.

In [ ]:
import os
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import Image
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay
from xgboost import XGBClassifier

RANDOM_STATE = 55

os.makedirs("../images", exist_ok=True)
os.makedirs("../models", exist_ok=True)

### Dataset
The data is the Heart Failure Prediction Dataset from Kaggle by fedesoriano, with 918 patients and 11 clinical measurements per patient. The target column HeartDisease is 1 when the patient has heart disease and 0 when they do not.

### Load the dataset
I load the heart disease CSV into a pandas DataFrame and check its shape and column names to confirm it is the right file.

In [ ]:
df = pd.read_csv("../data/heart.csv")
print(df.shape)
print(df.columns.tolist())

### Inspect the data
I look at the first few rows to see which columns hold text instead of numbers, since the models can only work with numbers.

In [ ]:
df.head()

### Explore the target
I check how many patients have heart disease, and whether patients with a cholesterol of 0 behave differently from the rest.

In [ ]:
print(df['HeartDisease'].value_counts())
print(df.groupby(df['Cholesterol'] == 0)['HeartDisease'].mean())

### Check for placeholder zeros
Some columns use 0 to mean a missing reading, so I count the zeros in Cholesterol and RestingBP.

In [ ]:
print((df['Cholesterol'] == 0).sum())
print((df['RestingBP'] == 0).sum())

### Remove impossible blood pressure
A resting blood pressure of 0 is not possible, so I drop that row from the data.

In [ ]:
df = df[df['RestingBP'] != 0].reset_index(drop=True)
print(df.shape)

### Flag missing cholesterol
Patients with a cholesterol of 0 have a much higher rate of heart disease, so I add a column that marks them before I replace the zeros.

In [ ]:
df['Cholesterol_missing'] = (df['Cholesterol'] == 0).astype(int)
print(df['Cholesterol_missing'].sum())

### Replace missing cholesterol
I replace the placeholder zeros with the median of the real cholesterol readings, which is less affected by extreme values than the mean.

In [ ]:
chol_median = df.loc[df['Cholesterol'] != 0, 'Cholesterol'].median()
df['Cholesterol'] = df['Cholesterol'].replace(0, chol_median)
print(chol_median)
print((df['Cholesterol'] == 0).sum())

### One-hot encoding
The 5 categorical columns hold text, so I convert each one into separate 0/1 columns, one per category, using pandas get_dummies.

In [ ]:
cat_variables = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

df = pd.get_dummies(data=df, prefix=cat_variables, columns=cat_variables, dtype=int)
df.head()

### Split the data
I separate the input features from the target column, then split the data 80/20 into a training set and a validation set to measure overfitting later.

In [ ]:
features = [x for x in df.columns if x != 'HeartDisease']
X_train, X_val, y_train, y_val = train_test_split(df[features], df['HeartDisease'], train_size=0.8, random_state=RANDOM_STATE)
print(X_train.shape, X_val.shape)

### Tuning helper
This function trains a model for each value of one hyperparameter, prints train and validation accuracy, and saves a plot of the two so I can see where the model starts to overfit.

In [ ]:
def tune(model_class, param_name, values, filename):
    train_acc = []
    val_acc = []

    for value in values:
        model = model_class(**{param_name: value}, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        train_acc.append(accuracy_score(y_train, model.predict(X_train)))
        val_acc.append(accuracy_score(y_val, model.predict(X_val)))

    table = pd.DataFrame({param_name: values, 'train': train_acc, 'validation': val_acc})
    print(table.round(4).to_string(index=False))

    plt.title('Train vs Validation accuracy')
    plt.xlabel(param_name)
    plt.ylabel('accuracy')
    plt.xticks(ticks=range(len(values)), labels=[str(v) for v in values])
    plt.plot(train_acc)
    plt.plot(val_acc)
    plt.legend(['Train', 'Validation'])
    plt.savefig("../images/" + filename)
    plt.close()
    return Image("../images/" + filename)

### Decision tree: min_samples_split
I train a decision tree for several values of min_samples_split and compare training accuracy against validation accuracy to see where the tree starts overfitting.

In [ ]:
min_samples_split_list = [2, 10, 30, 50, 100, 200, 300, 700]
tune(DecisionTreeClassifier, 'min_samples_split', min_samples_split_list, "tree_min_samples_split.png")

### Decision tree: max_depth
I train a decision tree for several values of max_depth and compare training accuracy against validation accuracy to find the depth where the tree stops improving on new data.

In [ ]:
max_depth_list = [1, 2, 3, 4, 8, 16, 32, 64, None]
tune(DecisionTreeClassifier, 'max_depth', max_depth_list, "tree_max_depth.png")

### Final decision tree
I combine the best max_depth and min_samples_split values from the two plots and measure train and validation accuracy for the final tree.

In [ ]:
decision_tree_model = DecisionTreeClassifier(min_samples_split=30, max_depth=4, random_state=RANDOM_STATE)
decision_tree_model.fit(X_train, y_train)

print(f"Metrics train:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_train), y_train):.4f}")
print(f"Metrics validation:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_val), y_val):.4f}")

### Random Forest: min_samples_split
I train a random forest for several values of min_samples_split and compare training accuracy against validation accuracy to see how a forest of trees overfits compared to a single tree.

In [ ]:
tune(RandomForestClassifier, 'min_samples_split', min_samples_split_list, "forest_min_samples_split.png")

### Random Forest: max_depth
I train a random forest for several values of max_depth and compare training accuracy against validation accuracy to see how depth affects overfitting inside a forest.

In [ ]:
tune(RandomForestClassifier, 'max_depth', max_depth_list, "forest_max_depth.png")

### Random Forest: n_estimators
I train a random forest with different numbers of trees and compare training accuracy against validation accuracy to see how many trees are worth using.

In [ ]:
n_estimators_list = [10, 50, 100, 500]
tune(RandomForestClassifier, 'n_estimators', n_estimators_list, "forest_n_estimators.png")

### Final Random Forest
I combine the chosen n_estimators, max_depth and min_samples_split values into one random forest and measure its train and validation accuracy. When two depths score the same on validation I take the shallower one, because it is simpler.

In [ ]:
random_forest_model = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=RANDOM_STATE)
random_forest_model.fit(X_train, y_train)

print(f"Metrics train:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_train), y_train):.4f}")
print(f"Metrics validation:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_val), y_val):.4f}")

### XGBoost eval set
XGBoost needs a separate set to decide when to stop training, so I split a small eval set out of the training data and leave the validation set untouched.

In [ ]:
X_train_fit, X_train_eval, y_train_fit, y_train_eval = train_test_split(X_train, y_train, train_size=0.8, random_state=RANDOM_STATE)
print(len(X_train_fit), len(X_train_eval))

### XGBoost
XGBoost trains trees one after another, where each new tree tries to fix the errors of the ones before it, and stops early once the eval metric stops improving.

In [ ]:
xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.1, early_stopping_rounds=10, random_state=RANDOM_STATE)
xgb_model.fit(X_train_fit, y_train_fit, eval_set=[(X_train_eval, y_train_eval)], verbose=True)

### Best XGBoost round
The best_iteration attribute holds the round with the lowest eval log loss, which is where early stopping measured the model at its best.

In [ ]:
xgb_model.best_iteration

### XGBoost accuracy
I measure the train and validation accuracy of the XGBoost model on the full training and validation sets so it can be compared fairly with the tree and the forest.

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_train), y_train):.4f}")
print(f"Metrics validation:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_val), y_val):.4f}")

### Cross-validation
A single validation split of 184 patients is noisy, so I score each model with 5-fold cross-validation on all the cleaned data. The hyperparameters were chosen using the validation split, so these scores are still slightly optimistic.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_models = {
    'Decision Tree': DecisionTreeClassifier(min_samples_split=30, max_depth=4, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=xgb_model.best_iteration + 1, learning_rate=0.1, random_state=RANDOM_STATE),
}

for name, model in cv_models.items():
    scores = cross_val_score(model, df[features], df['HeartDisease'], cv=cv, scoring='accuracy')
    print(f"{name}: mean accuracy {scores.mean():.4f}, std {scores.std():.4f}")

### Compare the models
I measure accuracy, precision, recall, F1 and ROC-AUC on the validation set for all three models. Recall matters most here, because it is the share of sick patients the model actually finds.

In [ ]:
models = {
    'Decision Tree': decision_tree_model,
    'Random Forest': random_forest_model,
    'XGBoost': xgb_model,
}

rows = []
for name, model in models.items():
    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)[:, 1]
    rows.append({
        'Model': name,
        'Train accuracy': accuracy_score(y_train, model.predict(X_train)),
        'Val accuracy': accuracy_score(y_val, predictions),
        'Precision': precision_score(y_val, predictions),
        'Recall': recall_score(y_val, predictions),
        'F1': f1_score(y_val, predictions),
        'ROC-AUC': roc_auc_score(y_val, probabilities),
    })

results = pd.DataFrame(rows).set_index('Model').round(4)
results

### Confusion matrices
I plot a confusion matrix for each model on the validation set to see how many sick patients each one misses and how many healthy patients it flags by mistake.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_val, y_val, ax=ax, colorbar=False)
    ax.set_title(name)
plt.savefig("../images/confusion_matrices.png")
plt.close()
Image("../images/confusion_matrices.png")

### Feature importance
I plot the ten features the random forest relies on most, to see which measurements drive its predictions.

In [ ]:
importances = pd.Series(random_forest_model.feature_importances_, index=features).sort_values().tail(10)

plt.figure(figsize=(8, 5))
plt.barh(importances.index, importances.values)
plt.title('Random Forest feature importance (top 10)')
plt.xlabel('importance')
plt.tight_layout()
plt.savefig("../images/feature_importance.png")
plt.close()
Image("../images/feature_importance.png")

### Save the best model
I pick the model with the highest validation recall, since missing a sick patient is the costliest mistake, and save it with joblib, together with the feature list and the cholesterol median, so new patients can be prepared in exactly the same way.

In [ ]:
best_name = results['Recall'].idxmax()
best_model = models[best_name]

joblib.dump({'model': best_model, 'features': features, 'cholesterol_median': chol_median}, "../models/heart_disease_model.joblib")
print("Saved model:", best_name)

### Predict for a new patient
I load the saved model and predict for one example patient, applying the same cleaning and encoding steps as the training data. This is a demo on made-up values and not medical advice.

In [ ]:
saved = joblib.load("../models/heart_disease_model.joblib")

new_patient = pd.DataFrame([{
    'Age': 54, 'Sex': 'M', 'ChestPainType': 'ASY', 'RestingBP': 140,
    'Cholesterol': 239, 'FastingBS': 0, 'RestingECG': 'Normal', 'MaxHR': 120,
    'ExerciseAngina': 'Y', 'Oldpeak': 1.5, 'ST_Slope': 'Flat',
}])

new_patient['Cholesterol_missing'] = (new_patient['Cholesterol'] == 0).astype(int)
new_patient['Cholesterol'] = new_patient['Cholesterol'].replace(0, saved['cholesterol_median'])
new_patient = pd.get_dummies(new_patient, dtype=int)
new_patient = new_patient.reindex(columns=saved['features'], fill_value=0)

prediction = saved['model'].predict(new_patient)[0]
probability = saved['model'].predict_proba(new_patient)[0, 1]
print("Prediction:", "heart disease" if prediction == 1 else "no heart disease")
print(f"Probability of heart disease: {probability:.2f}")

### Summary
In this project I cleaned a real heart disease dataset, flagged and filled the placeholder cholesterol values, and trained a Decision Tree, a Random Forest and an XGBoost model. The Random Forest had the best validation accuracy, recall and F1 and the best cross-validation accuracy, while XGBoost had the highest ROC-AUC, and the differences between the three are small on only 184 validation patients. The model with the best recall is saved with joblib and can predict for a new patient using the same preparation steps.